## 1. Setup & Imports

In [3]:
from datetime import datetime, timedelta
import time
import os
import warnings
warnings.filterwarnings('ignore')

In [4]:
# ── Path Configuration ────────────────────────────────────────────────
IN_COLAB = False  # Set to True when running on Google Colab

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/CS506/data"
else:
    DATA_ROOT = "../data"

CLEAN_DIR  = f"{DATA_ROOT}/clean"
NEWS_DIR   = f"{DATA_ROOT}/news"
CACHE_DIR  = f"{DATA_ROOT}/sentiment_cache"
MASTER_DIR = f"{DATA_ROOT}/master"

for _d in [CLEAN_DIR, NEWS_DIR, CACHE_DIR, MASTER_DIR]:
    os.makedirs(_d, exist_ok=True)

print(f"Running {'on Colab' if IN_COLAB else 'locally'}")
print(f"Data root: {DATA_ROOT}")

Running locally
Data root: ../data


## 2. Data Acquisition

In [5]:
import pandas as pd
import requests

def fetch_stock(symbol: str, start: datetime, end: datetime, clean_dir: str = "../data/clean"):
    path = f"{clean_dir}/{symbol}_clean.csv"
    if not os.path.exists(path):
        return {"s": "no_data"}

    df = pd.read_csv(path)
    df.rename(columns={"Date": "date"}, inplace=True)
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"] >= start) & (df["date"] < end)]

    if df.empty:
        return {"s": "no_data"}

    return {
        "s": "ok",
        "t": [int(d.timestamp()) for d in df["date"]],
        "c": df["Close"].tolist(),
    }


API_KEY = "d7crg5pr01qv03etebegd7crg5pr01qv03etebf0"

def fetch_news(symbol, start_date, end_date):
    url = f"https://finnhub.io/api/v1/company-news?symbol={symbol}&from={start_date}&to={end_date}&token={API_KEY}"
    response = requests.get(url)
    if response.status_code != 200:
        return []
    return response.json()

In [6]:
from transformers import pipeline

print("Downloading FinBERT ...")
finbert_pipe = pipeline("sentiment-analysis", model="ProsusAI/finbert")

def get_finbert_score(text):
    try:
        if not text or pd.isna(text):
            return 0.0
            
        result = finbert_pipe(str(text)[:512])[0]
        
        label = result['label']   
        score = result['score']    
        
        if label == 'positive':
            return score
        elif label == 'negative':
            return -score
        else:
            return 0.0
            
    except Exception as e:
        return 0.0

Device set to use cpu


In [8]:
START_DATE = datetime(2025, 4, 1)
END_DATE   = datetime(2026, 3, 31)
SYMBOLS = [
    "AAPL","AMZN","BAC","CVX","GOOG","GS","HD","JNJ","JPM",
    "MSFT","PFE","TSLA","UNH","XOM"
]

all_stocks_master = []

for symbol in SYMBOLS:
    cache_path = f"{CACHE_DIR}/{symbol}_processed.csv"

    if os.path.exists(cache_path):
        print(f"{symbol} already cached, skipping")
        all_stocks_master.append(pd.read_csv(cache_path))
        continue

    print(f"Processing {symbol} (news + stock + FinBERT) ...")

    try:
        news_path = f"{NEWS_DIR}/{symbol}_news.csv"
        if os.path.exists(news_path):
            df_news = pd.read_csv(news_path)
        else:
            all_news = []
            current = START_DATE
            while current < END_DATE:
                next_date = current + timedelta(days=7)
                news_data = fetch_news(symbol, current.strftime("%Y-%m-%d"), next_date.strftime("%Y-%m-%d"))
                for item in news_data:
                    all_news.append({
                        "date": datetime.fromtimestamp(item["datetime"]).strftime("%Y-%m-%d"),
                        "headline": item.get("headline", ""),
                        "summary": item.get("summary", "")
                    })
                current = next_date
                time.sleep(0.5)

            if not all_news:
                print(f"  {symbol}: no news found, skipping")
                continue

            df_news = pd.DataFrame(all_news)
            df_news.to_csv(news_path, index=False)
            print(f"  Saved {len(df_news)} news rows → {news_path}")

        print(f"  Running FinBERT ...")
        df_news['full_text'] = df_news['headline'].fillna('') + ". " + df_news['summary'].fillna('')
        df_news['sentiment_score'] = df_news['full_text'].apply(get_finbert_score)
        daily_sentiment = df_news.groupby('date')['sentiment_score'].mean().reset_index()

        stock_data = fetch_stock(symbol, START_DATE, END_DATE, clean_dir=CLEAN_DIR)
        if stock_data.get("s") != "ok":
            print(f"  {symbol}: stock data unavailable, skipping")
            continue

        df_stock = pd.DataFrame({
            "date": [datetime.fromtimestamp(t).strftime("%Y-%m-%d") for t in stock_data["t"]],
            "close": stock_data["c"]
        })

        df_final = pd.merge(df_stock, daily_sentiment, on="date", how="left").fillna(0)
        df_final["label"] = df_final["close"].diff().fillna(0).apply(lambda x: 1 if x > 0 else 0)
        df_final['sentiment_score_shifted'] = df_final['sentiment_score'].shift(1)
        df_final["symbol"] = symbol
        df_final = df_final.dropna()

        df_final.to_csv(cache_path, index=False)
        all_stocks_master.append(df_final)
        print(f"  {symbol} done, saved to cache")

    except Exception as e:
        print(f"  {symbol} error: {e}")

if all_stocks_master:
    mega_df = pd.concat(all_stocks_master, axis=0, ignore_index=True)
    mega_df.to_csv(f"{MASTER_DIR}/Mega_Stock_Dataset_Final.csv", index=False)
    print("\n" + "="*40)
    print(f"All done. {len(all_stocks_master)} stocks processed.")
    print(f"Total rows: {len(mega_df)}")
    print("="*40)

Processing AAPL (news + stock + FinBERT) ...
  Running FinBERT ...


KeyboardInterrupt: 

## 3. Feature Extraction

In [34]:
all_stocks_master = []

for symbol in SYMBOLS:
    cache_path = f"{CACHE_DIR}/{symbol}_mega_features.csv"

    if os.path.exists(cache_path):
        all_stocks_master.append(pd.read_csv(cache_path))
        continue

    print(f"Processing {symbol} ...")

    try:
        q_path = f"{CLEAN_DIR}/{symbol}_clean.csv"
        if not os.path.exists(q_path):
            print(f"  {symbol}_clean.csv not found, skipping")
            continue

        df_rich = pd.read_csv(q_path)
        df_rich.rename(columns={'Date': 'date'}, inplace=True)
        df_rich['date'] = pd.to_datetime(df_rich['date']).dt.strftime('%Y-%m-%d')

        merged = pd.merge(df_rich, daily_sentiment, on="date", how="left").fillna(0)

        cols_to_lag = ['Volume', 'Return', 'Market_Return', 'Excess_Return', 'Corr_30', 'Beta_30', 'sentiment_score']
        for col in cols_to_lag:
            if col in merged.columns:
                merged[f'{col}_lag1'] = merged[col].shift(1)

        merged["symbol"] = symbol
        merged = merged.dropna()

        merged.to_csv(cache_path, index=False)
        all_stocks_master.append(merged)
        print(f"  {symbol} merging complete")

    except Exception as e:
        print(f"  {symbol} failed: {e}")

if all_stocks_master:
    mega_df = pd.concat(all_stocks_master, axis=0, ignore_index=True)

    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    mega_df['symbol_id'] = le.fit_transform(mega_df['symbol'])

    mega_df.to_csv(f"{MASTER_DIR}/Final_Mega_Dataset.csv", index=False)
    print("\n" + "="*40)
    print(f"Done. Merged {len(all_stocks_master)} stocks.")
    print(f"Final dataset: {len(mega_df)} rows x {len(mega_df.columns)} columns")
    print("="*40)

Processing AMZN ...
  AMZN failed: name 'daily_sentiment' is not defined
Processing BAC ...
  BAC failed: name 'daily_sentiment' is not defined
Processing CVX ...
  CVX failed: name 'daily_sentiment' is not defined
Processing GS ...
  GS failed: name 'daily_sentiment' is not defined
Processing HD ...
  HD failed: name 'daily_sentiment' is not defined
Processing JNJ ...
  JNJ failed: name 'daily_sentiment' is not defined
Processing JPM ...
  JPM failed: name 'daily_sentiment' is not defined
Processing PFE ...
  PFE failed: name 'daily_sentiment' is not defined
Processing UNH ...
  UNH failed: name 'daily_sentiment' is not defined
Processing XOM ...
  XOM failed: name 'daily_sentiment' is not defined

Done. Merged 4 stocks.
Final dataset: 5028 rows x 25 columns


## 4. Model Training

In [28]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, f1_score
import plotly.express as px

df_mega = pd.read_csv(f"{MASTER_DIR}/Final_Mega_Dataset.csv")
if 'label' not in df_mega.columns:
    df_mega['label'] = (df_mega['Return'] > 0).astype(int)

df_mega = df_mega.dropna().reset_index(drop=True)

features = [
    'Volume_lag1', 'Return_lag1', 'Market_Return_lag1',
    'Excess_Return_lag1', 'Corr_30_lag1', 'Beta_30_lag1',
    'sentiment_score_lag1', 'symbol_id'
]

X = df_mega[features]
y = df_mega['label']

n = len(df_mega)
train_end = int(n * 0.7)
valid_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_valid = X.iloc[train_end:valid_end]
y_valid = y.iloc[train_end:valid_end]

X_test = X.iloc[valid_end:]
y_test = y.iloc[valid_end:]

print(f"Train size: {len(X_train)}")
print(f"Valid size: {len(X_valid)}")
print(f"Test size: {len(X_test)}")

Train size: 3519
Valid size: 754
Test size: 755


In [29]:
model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='logloss'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

y_valid_prob = model.predict_proba(X_valid)[:, 1]
y_valid_pred = (y_valid_prob >= 0.5).astype(int)

valid_accuracy = accuracy_score(y_valid, y_valid_pred)
valid_f1 = f1_score(y_valid, y_valid_pred)

print(f"Validation Accuracy: {valid_accuracy:.2%}")
print(f"Validation F1: {valid_f1:.4f}")

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=True)

fig = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title="Model Feature Importance",
    color='Importance',
    color_continuous_scale='Portland'
)
fig.update_layout(template='plotly_dark')
fig.show()

Validation Accuracy: 53.85%
Validation F1: 0.5896


In [78]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

thresholds = np.arange(0.53, 0.65, 0.01)

best_threshold = 0.5
best_f1 = -1

for t in thresholds:
    preds = (y_valid_prob >= t).astype(int)
    f1 = f1_score(y_valid, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best validation F1: {best_f1:.4f}")

Best threshold: 0.53
Best validation F1: 0.5764


In [79]:
param_candidates = [
    {"max_depth": 3, "learning_rate": 0.03, "min_child_weight": 3, "subsample": 0.8, "colsample_bytree": 0.8},
    {"max_depth": 4, "learning_rate": 0.03, "min_child_weight": 3, "subsample": 0.8, "colsample_bytree": 0.8},
    {"max_depth": 4, "learning_rate": 0.05, "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 0.8},
    {"max_depth": 5, "learning_rate": 0.03, "min_child_weight": 5, "subsample": 0.9, "colsample_bytree": 0.9},
]

results = []

for params in param_candidates:
    temp_model = xgb.XGBClassifier(
        n_estimators=1000,
        random_state=42,
        eval_metric='error',
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=5.0,
        **params
    )

    temp_model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    valid_prob = temp_model.predict_proba(X_valid)[:, 1]
    valid_pred = (valid_prob >= best_threshold).astype(int)

    acc = accuracy_score(y_valid, valid_pred)
    f1 = f1_score(y_valid, valid_pred)

    results.append((params, acc, f1, temp_model))

for params, acc, f1, _ in results:
    print(params, f"Accuracy={acc:.2%}", f"F1={f1:.4f}")

best_result = sorted(results, key=lambda x: x[2], reverse=True)[0]
best_params = best_result[0]
best_model_manual = best_result[3]

print("Best params:", best_params)

{'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 0.8} Accuracy=56.37% F1=0.5830
{'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 0.8} Accuracy=53.71% F1=0.5675
{'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8} Accuracy=52.92% F1=0.5557
{'max_depth': 5, 'learning_rate': 0.03, 'min_child_weight': 5, 'subsample': 0.9, 'colsample_bytree': 0.9} Accuracy=51.99% F1=0.5394
Best params: {'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 0.8}


## 5. Hyperparameter Tuning

In [ ]:
'''
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

cv = TimeSeriesSplit(n_splits=5)
grid_search = GridSearchCV(
    xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.2%}")

best_model = grid_search.best_estimator_
'''

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 100, 'subsample': 0.7}
Best CV accuracy: 53.89%


## 6. Evaluation

In [80]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_test_prob = best_model_manual.predict_proba(X_test)[:, 1]

for t in [0.50, best_threshold]:
    y_pred = (y_test_prob >= t).astype(int)

    print(f"\n===== Threshold = {t:.2f} =====")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
    print(f"F1: {f1_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))

importance_df_best = pd.DataFrame({
    'Feature': features,
    'Importance': best_model_manual.feature_importances_
}).sort_values(by='Importance', ascending=True)

fig = px.bar(
    importance_df_best,
    x='Importance',
    y='Feature',
    orientation='h',
    title="Best Manual-Tuned Model: Feature Importance",
    color='Importance',
    color_continuous_scale='Portland'
)
fig.update_layout(template='plotly_dark')
fig.show()


===== Threshold = 0.50 =====
Accuracy: 53.11%
F1: 0.5950
              precision    recall  f1-score   support

           0       0.53      0.38      0.44       369
           1       0.53      0.67      0.59       386

    accuracy                           0.53       755
   macro avg       0.53      0.53      0.52       755
weighted avg       0.53      0.53      0.52       755


===== Threshold = 0.53 =====
Accuracy: 53.91%
F1: 0.5756
              precision    recall  f1-score   support

           0       0.53      0.46      0.50       369
           1       0.54      0.61      0.58       386

    accuracy                           0.54       755
   macro avg       0.54      0.54      0.54       755
weighted avg       0.54      0.54      0.54       755

